# Video & Audio Emotion Recognition Integration

This notebook integrates both video (facial expressions) and audio (speech) emotion recognition models to provide comprehensive emotion analysis with:
- Segment-wise analysis (every 2 seconds)
- Individual scores for each segment
- Final aggregated scores
- Confidence levels and percentages for all emotions

## ⚠️ Prerequisites

Before running this notebook, you need to **train and save both models**:

1. **Video Model**: Train in `Vedio_emotion_Recognition.ipynb`
   - Saves to: `./models/best_video_emotion_model.h5`

2. **Audio Model**: Train in `Audio_Emotion_Recognition.ipynb`
   - Saves to: `./emotion-recognition-final/`

**If you haven't trained these models yet**, run those notebooks first!

## 1. Imports and Setup

In [2]:
# Standard library imports
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Core scientific computing
import numpy as np

# Check and import required packages with helpful error messages
missing_packages = []

try:
    import cv2
except ImportError:
    missing_packages.append("opencv-python")

try:
    import tensorflow as tf
except ImportError:
    missing_packages.append("tensorflow")

try:
    import librosa
except ImportError:
    missing_packages.append("librosa")

try:
    import soundfile
except ImportError:
    missing_packages.append("soundfile")

try:
    import torch
except ImportError:
    missing_packages.append("torch")

try:
    from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
except ImportError:
    missing_packages.append("transformers")

# Face detection uses OpenCV Haar cascades (no extra package needed)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    if 'matplotlib' not in missing_packages:
        missing_packages.append("matplotlib")
    if 'seaborn' not in missing_packages:
        missing_packages.append("seaborn")

# moviepy not required - we use ffmpeg (subprocess) to extract audio from video

# Display results
if missing_packages:
    print("❌ Missing packages detected!")
    print("\nPlease install the following packages:")
    print(f"   pip install {' '.join(missing_packages)}")
    print("\nOr install all requirements:")
    print("   pip install -r integration_requirements.txt")
else:
    print("✅ All imports successful!")
    print(f"\nTensorFlow version: {tf.__version__}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU Available (TF): {len(tf.config.list_physical_devices('GPU')) > 0}")
    print(f"GPU Available (PyTorch): {torch.cuda.is_available()}")

2026-02-02 04:49:53.096123: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 04:49:53.285886: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ All imports successful!

TensorFlow version: 2.20.0
PyTorch version: 2.10.0+cu128
GPU Available (TF): True
GPU Available (PyTorch): True


### Install Missing Packages (If Needed)

If you see missing packages above, run: `pip install <package_names>`.  
**Audio extraction** uses **ffmpeg** (not moviepy). Install ffmpeg on your system if needed, e.g. `sudo apt install ffmpeg` (Linux) or download from https://ffmpeg.org

In [3]:
# Uncomment and run this cell if you need to install packages
# !pip install librosa soundfile transformers
# !pip install tf_keras  # For loading models saved with older TensorFlow/Keras versions
# 
# Or install all requirements at once:
# !pip install -r integration_requirements.txt

## 2. Configuration

### Check if Models Exist

Run this cell to verify both models are available before proceeding:

In [4]:
# Emotion labels (must match both models)
EMOTIONS = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"
}
EMOTION_NAMES = list(EMOTIONS.values())

# Video model configuration
FRAMES_PER_VIDEO = 10
IMG_SIZE = 160
# Video model: same paths as Vedio_emotion_Recognition.ipynb
VIDEO_MODEL_CANDIDATES = [
    "./models/best_video_emotion_model.h5",
    "ML Models/models/best_video_emotion_model.h5",
    "models/best_video_emotion_model.h5",
]

# Audio model: same path as Audio_Emotion_Recognition.ipynb (emotion-recognition-final)
AUDIO_MODEL_CANDIDATES = [
    "./emotion-recognition-final",
    "ML Models/emotion-recognition-final",
    "emotion-recognition-final",
]

# Segment configuration
SEGMENT_DURATION_SEC = 2  # Analyze every 2 seconds

# Integration weights (adjust based on your preference)
VIDEO_WEIGHT = 0.6  # 60% weight to video
AUDIO_WEIGHT = 0.4  # 40% weight to audio

print("✅ Configuration loaded!")
print(f"Emotions: {EMOTION_NAMES}")
print(f"Segment duration: {SEGMENT_DURATION_SEC} seconds")
print(f"Weights - Video: {VIDEO_WEIGHT*100}%, Audio: {AUDIO_WEIGHT*100}%")

# Check if models exist
print("\nChecking for trained models...\n")

# Check video model
video_found = False
for path in VIDEO_MODEL_CANDIDATES:
    if Path(path).exists():
        print(f"✅ Video model found: {path}")
        video_found = True
        break

if not video_found:
    print("❌ Video model NOT FOUND!")
    print("   Expected locations:")
    for p in VIDEO_MODEL_CANDIDATES:
        print(f"   - {p}")
    print("\n   👉 Train the model in Vedio_emotion_Recognition.ipynb first!")

# Check audio model
print()
audio_found = False
for path in AUDIO_MODEL_CANDIDATES:
    if Path(path).exists():
        print(f"✅ Audio model found: {path}")
        audio_found = True
        break

if not audio_found:
    print("❌ Audio model NOT FOUND!")
    print("   Expected locations:")
    for p in AUDIO_MODEL_CANDIDATES:
        print(f"   - {p}")
    print("\n   👉 Train the model in Audio_Emotion_Recognition.ipynb first!")

print("\n" + "="*60)
if video_found and audio_found:
    print("✅ All models ready! You can proceed with the integration.")
else:
    print("⚠️  Please train the missing model(s) before continuing.")

✅ Configuration loaded!
Emotions: ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
Segment duration: 2 seconds
Weights - Video: 60.0%, Audio: 40.0%

Checking for trained models...

✅ Video model found: ./models/best_video_emotion_model.h5

✅ Audio model found: ./emotion-recognition-final

✅ All models ready! You can proceed with the integration.


In [5]:
# Emotion labels (must match both models)
EMOTIONS = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"
}
EMOTION_NAMES = list(EMOTIONS.values())

# Video model configuration
FRAMES_PER_VIDEO = 10
IMG_SIZE = 160
# Video model: same paths as Vedio_emotion_Recognition.ipynb
VIDEO_MODEL_CANDIDATES = [
    "./models/best_video_emotion_model.h5",
    "ML Models/models/best_video_emotion_model.h5",
    "models/best_video_emotion_model.h5",
]

# Audio model: same path as Audio_Emotion_Recognition.ipynb (emotion-recognition-final)
AUDIO_MODEL_CANDIDATES = [
    "./emotion-recognition-final",
    "ML Models/emotion-recognition-final",
    "emotion-recognition-final",
]

# Segment configuration
SEGMENT_DURATION_SEC = 2  # Analyze every 2 seconds

# Integration weights (adjust based on your preference)
VIDEO_WEIGHT = 0.6  # 60% weight to video
AUDIO_WEIGHT = 0.4  # 40% weight to audio

print("✅ Configuration loaded!")
print(f"Emotions: {EMOTION_NAMES}")
print(f"Segment duration: {SEGMENT_DURATION_SEC} seconds")
print(f"Weights - Video: {VIDEO_WEIGHT*100}%, Audio: {AUDIO_WEIGHT*100}%")

✅ Configuration loaded!
Emotions: ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
Segment duration: 2 seconds
Weights - Video: 60.0%, Audio: 40.0%


## 3. Video Emotion Recognition Functions

In [6]:
def load_video_model(model_path=None):
    """Load the trained video model (same as Vedio_emotion_Recognition.ipynb)."""
    candidates = list(VIDEO_MODEL_CANDIDATES)
    if model_path:
        candidates.insert(0, model_path)
    path = None
    for p in candidates:
        if Path(p).exists():
            path = Path(p)
            break
    if path is None:
        raise FileNotFoundError("Video model .h5 not found. Tried: " + ", ".join(candidates))
    
    # Use tf_keras for compatibility with older models containing tf_keras layers
    import os
    os.environ['TF_USE_LEGACY_KERAS'] = '1'  # Force use of tf_keras
    
    try:
        # Try with tf_keras (for models saved with TF 2.x legacy Keras)
        import tf_keras
        model = tf_keras.models.load_model(str(path), compile=False)
        print(f"✅ Video model loaded from {path} (using tf_keras)")
        return model
    except (ImportError, Exception) as e:
        # Fallback to regular keras
        try:
            model = tf.keras.models.load_model(str(path), compile=False, safe_mode=False)
        except TypeError:
            model = tf.keras.models.load_model(str(path), compile=False)
        print(f"✅ Video model loaded from {path}")
        return model


# ImageNet normalization (same as Vedio_emotion_Recognition.ipynb)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# OpenCV Haar cascade face detector (built-in, no extra packages, avoids MediaPipe/RetinaFace issues)
_face_cascade = None

def detect_and_crop_face(frame):
    """Detect and crop face using OpenCV Haar cascade (built-in, no TF/MediaPipe conflicts)."""
    global _face_cascade
    if _face_cascade is None:
        _face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w = frame.shape[:2]
    faces = _face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(30, 30))
    if len(faces) > 0:
        x, y, fw, fh = faces[0]
        padding = 20
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(w, x + fw + padding)
        y2 = min(h, y + fh + padding)
        if x2 > x1 and y2 > y1:
            return frame[y1:y2, x1:x2]
    # Fallback: center crop
    size = min(h, w)
    y1, x1 = (h - size) // 2, (w - size) // 2
    return frame[y1:y1+size, x1:x1+size]


def extract_frames_from_segment(video_path, start_frame, end_frame, num_frames, target_size):
    """Extract frames with same preprocessing as Vedio_emotion_Recognition.ipynb."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if end_frame > total:
        end_frame = total
    if start_frame >= end_frame:
        cap.release()
        return None
    indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frame = detect_and_crop_face(frame)
            frame = cv2.resize(frame, target_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        else:
            if frames:
                frames.append(frames[-1].copy())
            else:
                frames.append(np.zeros((*target_size, 3), dtype=np.uint8))
    cap.release()
    if len(frames) == 0:
        return None
    frames = np.array(frames, dtype=np.float32) / 255.0
    frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
    return frames


def predict_video_segments(video_path, model, segment_duration=SEGMENT_DURATION_SEC):
    """Predict emotions for each segment of the video."""
    video_path = Path(video_path)
    
    # Get video properties
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    segment_length_frames = max(1, int(round(fps * segment_duration)))
    n_segments = max(1, (total_frames + segment_length_frames - 1) // segment_length_frames)
    
    print(f"📹 Video Analysis:")
    print(f"   FPS: {fps:.2f}, Total frames: {total_frames}")
    print(f"   Segments: {n_segments} ({segment_duration}s each)")
    
    segment_results = []
    
    for seg in range(n_segments):
        start_frame = seg * segment_length_frames
        end_frame = min(start_frame + segment_length_frames, total_frames)
        
        if start_frame >= end_frame:
            continue
        
        start_time = start_frame / fps
        end_time = end_frame / fps
        
        # Extract frames for this segment
        frames = extract_frames_from_segment(
            video_path, start_frame, end_frame, 
            FRAMES_PER_VIDEO, (IMG_SIZE, IMG_SIZE)
        )
        
        if frames is None:
            print(f"   ⚠️  Segment {seg + 1}: No frames extracted")
            continue
        
        # Predict
        batch = np.expand_dims(frames, axis=0)
        probs = model.predict(batch, verbose=0)[0]
        
        predicted_idx = int(np.argmax(probs))
        confidence = float(probs[predicted_idx])
        
        segment_results.append({
            'segment': seg + 1,
            'start_time': start_time,
            'end_time': end_time,
            'probabilities': probs,
            'predicted_emotion': EMOTION_NAMES[predicted_idx],
            'confidence': confidence
        })
        
        print(f"   ✓ Segment {seg + 1}/{n_segments} ({start_time:.1f}s-{end_time:.1f}s): "
              f"{EMOTION_NAMES[predicted_idx]} ({confidence*100:.1f}%)")
    
    return segment_results

print("✅ Video emotion recognition functions loaded!")

✅ Video emotion recognition functions loaded!


## 4. Audio Emotion Recognition Functions

In [7]:
def load_audio_model(model_path=None):
    """Load the trained audio model (same as Audio_Emotion_Recognition.ipynb)."""
    candidates = list(AUDIO_MODEL_CANDIDATES)
    if model_path:
        candidates.insert(0, model_path)
    path = None
    for p in candidates:
        if Path(p).exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError("Audio model not found. Tried: " + ", ".join(candidates))
    feature_extractor = AutoFeatureExtractor.from_pretrained(path)
    model = AutoModelForAudioClassification.from_pretrained(path)
    model.eval()
    with open(f"{path}/label_mapping.json", "r") as f:
        label_mapping = json.load(f)
    print(f"✅ Audio model loaded from {path}")
    return feature_extractor, model, label_mapping


def extract_audio_from_video(video_path, output_path="temp_audio.wav"):
    """Extract audio from video using ffmpeg (no moviepy required)."""
    import subprocess
    import shutil
    
    if shutil.which("ffmpeg") is None:
        raise RuntimeError(
            "ffmpeg not found! Install ffmpeg to extract audio from video.\n"
            "  Linux: sudo apt install ffmpeg\n"
            "  macOS: brew install ffmpeg\n"
            "  Windows: Download from https://ffmpeg.org"
        )
    
    video_path = str(Path(video_path).resolve())
    output_path = str(Path(output_path).resolve())
    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("ffmpeg failed. Is ffmpeg installed? Error: " + (result.stderr or result.stdout or ""))
    if not Path(output_path).exists():
        raise ValueError("No audio track in video or ffmpeg produced no file")
    return output_path


def predict_audio_segments(audio_path, feature_extractor, model, label_mapping, segment_duration=SEGMENT_DURATION_SEC):
    """Predict emotions for each segment of the audio."""
    # Load full audio
    audio, sr = librosa.load(audio_path, sr=feature_extractor.sampling_rate)
    
    total_duration = len(audio) / sr
    n_segments = max(1, int(np.ceil(total_duration / segment_duration)))
    
    print(f"\n🎵 Audio Analysis:")
    print(f"   Sample rate: {sr} Hz, Duration: {total_duration:.2f}s")
    print(f"   Segments: {n_segments} ({segment_duration}s each)")
    
    segment_results = []
    
    for seg in range(n_segments):
        start_time = seg * segment_duration
        end_time = min((seg + 1) * segment_duration, total_duration)
        
        start_sample = int(start_time * sr)
        end_sample = int(end_time * sr)
        
        # Extract segment
        audio_segment = audio[start_sample:end_sample]
        
        if len(audio_segment) == 0:
            continue
        
        # Preprocess
        inputs = feature_extractor(
            audio_segment,
            sampling_rate=sr,
            return_tensors="pt",
            padding=True,
            max_length=16000 * 5,
            truncation=True
        )
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # Get probabilities
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]
        
        # Convert to numpy and create probability array matching EMOTION_NAMES order
        probs = np.zeros(len(EMOTION_NAMES))
        for idx, prob in enumerate(probabilities):
            emotion = label_mapping["id2label"][str(idx)]
            if emotion in EMOTION_NAMES:
                emotion_idx = EMOTION_NAMES.index(emotion)
                probs[emotion_idx] = prob.item()
        
        predicted_idx = int(np.argmax(probs))
        confidence = float(probs[predicted_idx])
        
        segment_results.append({
            'segment': seg + 1,
            'start_time': start_time,
            'end_time': end_time,
            'probabilities': probs,
            'predicted_emotion': EMOTION_NAMES[predicted_idx],
            'confidence': confidence
        })
        
        print(f"   ✓ Segment {seg + 1}/{n_segments} ({start_time:.1f}s-{end_time:.1f}s): "
              f"{EMOTION_NAMES[predicted_idx]} ({confidence*100:.1f}%)")
    
    return segment_results

print("✅ Audio emotion recognition functions loaded!")

✅ Audio emotion recognition functions loaded!


## 5. Integration and Fusion Functions

In [8]:
def integrate_video_audio_segments(video_results, audio_results, video_weight=VIDEO_WEIGHT, audio_weight=AUDIO_WEIGHT):
    """
    Integrate video and audio predictions for each segment using weighted fusion.
    """
    # Ensure we have the same number of segments (or handle mismatch)
    min_segments = min(len(video_results), len(audio_results))
    
    integrated_results = []
    
    for i in range(min_segments):
        video_seg = video_results[i]
        audio_seg = audio_results[i]
        
        # Weighted fusion of probabilities
        fused_probs = (video_weight * video_seg['probabilities'] + 
                       audio_weight * audio_seg['probabilities'])
        
        predicted_idx = int(np.argmax(fused_probs))
        confidence = float(fused_probs[predicted_idx])
        
        integrated_results.append({
            'segment': i + 1,
            'start_time': video_seg['start_time'],
            'end_time': video_seg['end_time'],
            'video_emotion': video_seg['predicted_emotion'],
            'video_confidence': video_seg['confidence'],
            'audio_emotion': audio_seg['predicted_emotion'],
            'audio_confidence': audio_seg['confidence'],
            'fused_probabilities': fused_probs,
            'fused_emotion': EMOTION_NAMES[predicted_idx],
            'fused_confidence': confidence
        })
    
    return integrated_results


def calculate_overall_scores(segment_results):
    """
    Calculate overall emotion scores by averaging across all segments.
    """
    if not segment_results:
        return None
    
    # Average probabilities across all segments
    all_probs = np.array([seg['fused_probabilities'] for seg in segment_results])
    overall_probs = np.mean(all_probs, axis=0)
    
    # Get final prediction
    final_idx = int(np.argmax(overall_probs))
    final_emotion = EMOTION_NAMES[final_idx]
    final_confidence = float(overall_probs[final_idx])
    
    # Create emotion percentage dictionary
    emotion_percentages = {
        emotion: float(overall_probs[i] * 100)
        for i, emotion in enumerate(EMOTION_NAMES)
    }
    
    return {
        'final_emotion': final_emotion,
        'final_confidence': final_confidence,
        'overall_probabilities': overall_probs,
        'emotion_percentages': emotion_percentages
    }

print("✅ Integration functions loaded!")

✅ Integration functions loaded!


## 6. Visualization Functions

In [9]:
def display_segment_results(segment_results):
    """Display detailed results for each segment."""
    print("\n" + "="*80)
    print("SEGMENT-WISE ANALYSIS")
    print("="*80)
    
    for seg in segment_results:
        print(f"\n📍 Segment {seg['segment']} ({seg['start_time']:.1f}s - {seg['end_time']:.1f}s)")
        print("-" * 80)
        print(f"   📹 Video:  {seg['video_emotion']:12s} (Confidence: {seg['video_confidence']*100:5.1f}%)")
        print(f"   🎵 Audio:  {seg['audio_emotion']:12s} (Confidence: {seg['audio_confidence']*100:5.1f}%)")
        print(f"   🔀 Fused:  {seg['fused_emotion']:12s} (Confidence: {seg['fused_confidence']*100:5.1f}%)")
        
        # Show top 3 emotions for fused result
        sorted_indices = np.argsort(seg['fused_probabilities'])[::-1][:3]
        print(f"\n   Top 3 Emotions (Fused):")
        for i, idx in enumerate(sorted_indices, 1):
            emotion = EMOTION_NAMES[idx]
            prob = seg['fused_probabilities'][idx] * 100
            bar_length = int(prob / 2)  # Scale to 50 chars max
            bar = "█" * bar_length
            print(f"      {i}. {emotion:12s} [{bar:<50s}] {prob:5.1f}%")


def display_overall_results(overall_scores):
    """Display overall aggregated results."""
    print("\n" + "="*80)
    print("OVERALL RESULTS (Averaged Across All Segments)")
    print("="*80)
    
    print(f"\n🎯 Final Emotion: {overall_scores['final_emotion'].upper()}")
    print(f"📊 Confidence: {overall_scores['final_confidence']*100:.2f}%")
    
    print("\n" + "-"*80)
    print("Emotion Percentages:")
    print("-"*80)
    
    # Sort emotions by percentage
    sorted_emotions = sorted(
        overall_scores['emotion_percentages'].items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    for emotion, percentage in sorted_emotions:
        bar_length = int(percentage / 2)  # Scale to 50 chars max
        bar = "█" * bar_length + "░" * (50 - bar_length)
        print(f"{emotion:12s} [{bar}] {percentage:5.2f}%")
    
    print("="*80)


def plot_emotion_timeline(segment_results):
    """Plot emotion predictions over time."""
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    segments = [seg['segment'] for seg in segment_results]
    times = [(seg['start_time'] + seg['end_time']) / 2 for seg in segment_results]
    
    # Video emotions
    video_emotions = [seg['video_emotion'] for seg in segment_results]
    video_confidences = [seg['video_confidence'] * 100 for seg in segment_results]
    
    axes[0].bar(segments, video_confidences, color='steelblue', alpha=0.7)
    axes[0].set_ylabel('Confidence (%)', fontsize=12)
    axes[0].set_title('Video Emotion Recognition', fontsize=14, fontweight='bold')
    axes[0].set_ylim(0, 100)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add emotion labels
    for i, (seg, emotion, conf) in enumerate(zip(segments, video_emotions, video_confidences)):
        axes[0].text(seg, conf + 2, emotion, ha='center', va='bottom', fontsize=9, rotation=45)
    
    # Audio emotions
    audio_emotions = [seg['audio_emotion'] for seg in segment_results]
    audio_confidences = [seg['audio_confidence'] * 100 for seg in segment_results]
    
    axes[1].bar(segments, audio_confidences, color='coral', alpha=0.7)
    axes[1].set_ylabel('Confidence (%)', fontsize=12)
    axes[1].set_title('Audio Emotion Recognition', fontsize=14, fontweight='bold')
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis='y', alpha=0.3)
    
    for i, (seg, emotion, conf) in enumerate(zip(segments, audio_emotions, audio_confidences)):
        axes[1].text(seg, conf + 2, emotion, ha='center', va='bottom', fontsize=9, rotation=45)
    
    # Fused emotions
    fused_emotions = [seg['fused_emotion'] for seg in segment_results]
    fused_confidences = [seg['fused_confidence'] * 100 for seg in segment_results]
    
    axes[2].bar(segments, fused_confidences, color='mediumseagreen', alpha=0.7)
    axes[2].set_xlabel('Segment Number', fontsize=12)
    axes[2].set_ylabel('Confidence (%)', fontsize=12)
    axes[2].set_title('Fused Emotion Recognition (Video + Audio)', fontsize=14, fontweight='bold')
    axes[2].set_ylim(0, 100)
    axes[2].grid(axis='y', alpha=0.3)
    
    for i, (seg, emotion, conf) in enumerate(zip(segments, fused_emotions, fused_confidences)):
        axes[2].text(seg, conf + 2, emotion, ha='center', va='bottom', fontsize=9, rotation=45)
    
    plt.tight_layout()
    plt.show()


def plot_emotion_distribution(overall_scores):
    """Plot overall emotion distribution."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    emotions = list(overall_scores['emotion_percentages'].keys())
    percentages = list(overall_scores['emotion_percentages'].values())
    
    # Bar chart
    colors = plt.cm.Set3(range(len(emotions)))
    bars = ax1.bar(emotions, percentages, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_ylabel('Percentage (%)', fontsize=12)
    ax1.set_xlabel('Emotion', fontsize=12)
    ax1.set_title('Overall Emotion Distribution', fontsize=14, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
    
    # Pie chart
    ax2.pie(percentages, labels=emotions, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 10})
    ax2.set_title('Emotion Proportion', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions loaded!")

✅ Visualization functions loaded!


## 7. Main Integration Pipeline

In [10]:
def analyze_video_with_audio(video_path, 
                             video_model_path=None,
                             audio_model_path=None,
                             segment_duration=SEGMENT_DURATION_SEC,
                             video_weight=VIDEO_WEIGHT,
                             audio_weight=AUDIO_WEIGHT,
                             show_plots=True):
    """
    Complete pipeline to analyze a video with both video and audio emotion recognition.
    
    Parameters:
    -----------
    video_path : str or Path
        Path to the video file
    video_model_path : str
        Path to the trained video model
    audio_model_path : str
        Path to the trained audio model
    segment_duration : float
        Duration of each segment in seconds
    video_weight : float
        Weight for video predictions (0-1)
    audio_weight : float
        Weight for audio predictions (0-1)
    show_plots : bool
        Whether to display visualization plots
    
    Returns:
    --------
    dict : Complete analysis results
    """
    print("="*80)
    print("VIDEO & AUDIO EMOTION RECOGNITION INTEGRATION")
    print("="*80)
    print(f"\n📁 Video: {video_path}")
    print(f"⚖️  Weights: Video={video_weight*100}%, Audio={audio_weight*100}%")
    print(f"⏱️  Segment Duration: {segment_duration}s\n")
    
    # Load models
    print("🔄 Loading models...")
    video_model = load_video_model(video_model_path)
    feature_extractor, audio_model, label_mapping = load_audio_model(audio_model_path)
    
    # Extract audio from video
    print("\n🔄 Extracting audio from video...")
    temp_audio_path = "temp_audio.wav"
    try:
        extract_audio_from_video(video_path, temp_audio_path)
        print(f"✅ Audio extracted to {temp_audio_path}")
    except Exception as e:
        print(f"❌ Error extracting audio: {e}")
        return None
    
    # Analyze video
    print("\n" + "="*80)
    video_results = predict_video_segments(video_path, video_model, segment_duration)
    
    # Analyze audio
    print("\n" + "="*80)
    audio_results = predict_audio_segments(
        temp_audio_path, feature_extractor, audio_model, 
        label_mapping, segment_duration
    )
    
    # Integrate results
    print("\n" + "="*80)
    print("🔀 Integrating video and audio predictions...")
    integrated_results = integrate_video_audio_segments(
        video_results, audio_results, video_weight, audio_weight
    )
    print(f"✅ Integrated {len(integrated_results)} segments")
    
    # Calculate overall scores
    overall_scores = calculate_overall_scores(integrated_results)
    
    # Display results
    display_segment_results(integrated_results)
    display_overall_results(overall_scores)
    
    # Visualizations
    if show_plots:
        print("\n📊 Generating visualizations...")
        plot_emotion_timeline(integrated_results)
        plot_emotion_distribution(overall_scores)
    
    # Clean up temporary audio file
    if Path(temp_audio_path).exists():
        os.remove(temp_audio_path)
        print(f"\n🧹 Cleaned up temporary file: {temp_audio_path}")
    
    # Return complete results
    return {
        'video_results': video_results,
        'audio_results': audio_results,
        'integrated_results': integrated_results,
        'overall_scores': overall_scores
    }

print("✅ Main pipeline function loaded!")

✅ Main pipeline function loaded!


## 8. Example Usage

In [11]:
# Example: Analyze a video file
# Replace with your actual video path

video_path = "path/to/your/video.mp4"

# Run the complete analysis
results = analyze_video_with_audio(
    video_path=video_path,
    segment_duration=2,  # Analyze every 2 seconds
    video_weight=0.5,    # 60% weight to video
    audio_weight=0.5,    # 40% weight to audio
    show_plots=True
)

VIDEO & AUDIO EMOTION RECOGNITION INTEGRATION

📁 Video: path/to/your/video.mp4
⚖️  Weights: Video=50.0%, Audio=50.0%
⏱️  Segment Duration: 2s

🔄 Loading models...


I0000 00:00:1770000608.792310     755 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


✅ Video model loaded from models/best_video_emotion_model.h5 (using tf_keras)


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

✅ Audio model loaded from ./emotion-recognition-final

🔄 Extracting audio from video...
❌ Error extracting audio: ffmpeg failed. Is ffmpeg installed? Error: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-l

## 9. Access Individual Results

In [12]:
# Access specific results if needed

# Video-only results
# video_results = results['video_results']

# Audio-only results
# audio_results = results['audio_results']

# Integrated segment results
# integrated_results = results['integrated_results']

# Overall aggregated scores
# overall_scores = results['overall_scores']

# Example: Print final emotion and confidence
# print(f"Final Emotion: {overall_scores['final_emotion']}")
# print(f"Confidence: {overall_scores['final_confidence']*100:.2f}%")

# Example: Print all emotion percentages
# for emotion, percentage in overall_scores['emotion_percentages'].items():
#     print(f"{emotion}: {percentage:.2f}%")

## 10. Advanced: Custom Analysis Functions

In [13]:
def export_results_to_json(results, output_path="emotion_analysis_results.json"):
    """Export analysis results to JSON file."""
    # Convert numpy arrays to lists for JSON serialization
    export_data = {
        'integrated_results': [
            {
                'segment': seg['segment'],
                'start_time': seg['start_time'],
                'end_time': seg['end_time'],
                'video_emotion': seg['video_emotion'],
                'video_confidence': seg['video_confidence'],
                'audio_emotion': seg['audio_emotion'],
                'audio_confidence': seg['audio_confidence'],
                'fused_emotion': seg['fused_emotion'],
                'fused_confidence': seg['fused_confidence'],
                'fused_probabilities': seg['fused_probabilities'].tolist()
            }
            for seg in results['integrated_results']
        ],
        'overall_scores': {
            'final_emotion': results['overall_scores']['final_emotion'],
            'final_confidence': results['overall_scores']['final_confidence'],
            'emotion_percentages': results['overall_scores']['emotion_percentages'],
            'overall_probabilities': results['overall_scores']['overall_probabilities'].tolist()
        }
    }
    
    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"✅ Results exported to {output_path}")


def get_emotion_summary(results):
    """Get a concise summary of the emotion analysis."""
    overall = results['overall_scores']
    
    # Get top 3 emotions
    sorted_emotions = sorted(
        overall['emotion_percentages'].items(),
        key=lambda x: x[1],
        reverse=True
    )[:3]
    
    summary = {
        'primary_emotion': overall['final_emotion'],
        'confidence': f"{overall['final_confidence']*100:.2f}%",
        'top_3_emotions': [
            {
                'emotion': emotion,
                'percentage': f"{percentage:.2f}%"
            }
            for emotion, percentage in sorted_emotions
        ],
        'total_segments': len(results['integrated_results']),
        'segment_duration': f"{SEGMENT_DURATION_SEC}s"
    }
    
    return summary


def compare_video_audio_agreement(results):
    """Analyze agreement between video and audio predictions."""
    integrated = results['integrated_results']
    
    agreements = 0
    disagreements = 0
    
    for seg in integrated:
        if seg['video_emotion'] == seg['audio_emotion']:
            agreements += 1
        else:
            disagreements += 1
    
    total = len(integrated)
    agreement_rate = (agreements / total * 100) if total > 0 else 0
    
    print("\n" + "="*80)
    print("VIDEO-AUDIO AGREEMENT ANALYSIS")
    print("="*80)
    print(f"\n✅ Agreements: {agreements}/{total} ({agreement_rate:.1f}%)")
    print(f"❌ Disagreements: {disagreements}/{total} ({100-agreement_rate:.1f}%)")
    
    # Show disagreement details
    if disagreements > 0:
        print("\nDisagreement Details:")
        print("-"*80)
        for seg in integrated:
            if seg['video_emotion'] != seg['audio_emotion']:
                print(f"  Segment {seg['segment']} ({seg['start_time']:.1f}s-{seg['end_time']:.1f}s):")
                print(f"    Video: {seg['video_emotion']:12s} ({seg['video_confidence']*100:.1f}%)")
                print(f"    Audio: {seg['audio_emotion']:12s} ({seg['audio_confidence']*100:.1f}%)")
                print(f"    Fused: {seg['fused_emotion']:12s} ({seg['fused_confidence']*100:.1f}%)")
    
    return {
        'agreements': agreements,
        'disagreements': disagreements,
        'agreement_rate': agreement_rate
    }

print("✅ Advanced analysis functions loaded!")

✅ Advanced analysis functions loaded!


## 11. Example: Using Advanced Functions

In [14]:
# After running the analysis, you can use these advanced functions:

# Get a concise summary
# summary = get_emotion_summary(results)
# print(json.dumps(summary, indent=2))

# Compare video-audio agreement
# agreement_stats = compare_video_audio_agreement(results)

# Export results to JSON
# export_results_to_json(results, "my_video_analysis.json")

## 12. Interactive Widget (Optional)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, Video
    
    # File upload widget
    file_upload = widgets.FileUpload(
        accept='.mp4,.avi,.mov,.mkv',
        multiple=False,
        description='Upload Video'
    )
    
    # Video player area - shows the uploaded video
    video_player_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px'))
    
    def on_file_upload(change):
        """Display video player when a file is uploaded."""
        with video_player_output:
            clear_output()
            if not change.get('new'):
                return
            val = change['new']
            if not val:
                return
            try:
                if isinstance(val, (list, tuple)):
                    uploaded_file = val[0]
                    content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content
                else:
                    uploaded_file = list(val.values())[0]
                    content = uploaded_file['content']
                content = content.tobytes() if hasattr(content, 'tobytes') else content
                preview_path = Path("temp_preview_video.mp4")
                with open(preview_path, 'wb') as f:
                    f.write(content)
                display(Video(str(preview_path.resolve()), embed=True, width=640))
            except Exception as e:
                print(f"Could not preview video: {e}")
    
    file_upload.observe(on_file_upload, names='value')
    
    # Segment duration slider
    segment_slider = widgets.FloatSlider(
        value=2.0,
        min=1.0,
        max=5.0,
        step=0.5,
        description='Segment (s):',
        continuous_update=False
    )
    
    # Video weight slider
    video_weight_slider = widgets.FloatSlider(
        value=0.6,
        min=0.0,
        max=1.0,
        step=0.1,
        description='Video Weight:',
        continuous_update=False
    )
    
    # Analyze button
    analyze_btn = widgets.Button(
        description='Analyze Video',
        button_style='success',
        icon='play'
    )
    
    # Output area
    output = widgets.Output()
    
    def on_analyze_click(b):
        with output:
            clear_output()
            
            if not file_upload.value:
                print("❌ Please upload a video first!")
                return
            
            if 'analyze_video_with_audio' not in globals():
                print("❌ Please run all cells above first (Run All or run cells 1-18)!")
                print("   The analyze_video_with_audio function is not defined yet.")
                return
            
            # Save uploaded file (compatible with ipywidgets 7.x, 8.x, and tuple variants)
            val = file_upload.value
            if isinstance(val, (list, tuple)):
                # ipywidgets 8.x: value is list/tuple of {content, name, type, ...}
                uploaded_file = val[0]
                content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content
            else:
                # ipywidgets 7.x: value is dict {filename: {content, metadata}}
                uploaded_file = list(val.values())[0]
                content = uploaded_file['content']
            content = content.tobytes() if hasattr(content, 'tobytes') else content
            
            video_path = Path("temp_uploaded_video.mp4")
            with open(video_path, 'wb') as f:
                f.write(content)
            
            # Show video player in output
            display(Video(str(video_path.resolve()), embed=True, width=640))
            print("\n" + "="*60 + "\n")
            
            try:
                # Run analysis
                video_weight = video_weight_slider.value
                audio_weight = 1.0 - video_weight
                
                results = analyze_video_with_audio(
                    video_path=video_path,
                    segment_duration=segment_slider.value,
                    video_weight=video_weight,
                    audio_weight=audio_weight,
                    show_plots=True
                )
                
                # Show additional analysis (only if analysis succeeded)
                if results is not None:
                    compare_video_audio_agreement(results)
                else:
                    print("\n⚠️  Analysis could not complete (e.g. ffmpeg not found). See errors above.")
                
            except Exception as e:
                print(f"❌ Error during analysis: {e}")
                import traceback
                traceback.print_exc()
            
            finally:
                # Clean up
                if video_path.exists():
                    os.remove(video_path)
    
    analyze_btn.on_click(on_analyze_click)
    
    # Display UI
    print("✅ Interactive widget loaded!")
    print("\nUpload a video and click 'Analyze Video' to start:")
    display(widgets.VBox([
        widgets.HTML("<b>📁 Upload Video</b>"),
        file_upload,
        widgets.HTML("<b>▶️ Video Preview</b>"),
        video_player_output,
        widgets.HTML("<b>⚙️ Settings</b>"),
        segment_slider,
        video_weight_slider,
        analyze_btn,
        widgets.HTML("<b>📊 Analysis Output</b>"),
        output
    ]))
    
except ImportError:
    print("⚠️  ipywidgets not available. Use the analyze_video_with_audio() function directly.")

## Summary

This notebook provides a complete integration of video and audio emotion recognition with the following features:

### ✨ Key Features:

1. **Segment-wise Analysis**: Analyzes emotions every N seconds (default: 2 seconds)
2. **Dual Modality**: Combines both facial expressions (video) and speech (audio)
3. **Weighted Fusion**: Adjustable weights for video and audio predictions
4. **Detailed Output**: Shows predictions for each segment with confidence scores
5. **Overall Aggregation**: Final emotion scores averaged across all segments
6. **Rich Visualizations**: Timeline plots and distribution charts
7. **Agreement Analysis**: Compares video and audio predictions
8. **Export Capability**: Save results to JSON for further analysis

### 📊 Output Information:

For each segment, you get:
- Video emotion prediction + confidence
- Audio emotion prediction + confidence  
- Fused emotion prediction + confidence
- Probability distribution across all emotions

At the end, you get:
- Final overall emotion
- Final confidence score
- Percentage for each emotion (averaged across segments)
- Visual plots showing emotion timeline and distribution

### 🎯 Usage:

```python
results = analyze_video_with_audio(
    video_path="your_video.mp4",
    segment_duration=2,
    video_weight=0.6,
    audio_weight=0.4
)
```

### 📝 Notes:

- Make sure both models are trained and saved at the specified paths
- Video model: `./models/best_video_emotion_model.h5`
- Audio model: `./emotion-recognition-final/`
- Adjust weights based on which modality you trust more
- Segment duration can be adjusted (1-5 seconds recommended)